# BIRCH

## 1. Introduction
BIRCH incrementally summarizes data in a clustering-feature tree, making it useful for larger datasets. The threshold controls subcluster granularity and an optional final clustering stage combines subclusters.

**Why / when:** Select the method based on data geometry, scale, operational constraints, and the question being asked. This lesson uses deterministic data and runs offline.


## 2. Mathematical intuition
BIRCH incrementally summarizes data in a clustering-feature tree, making it useful for larger datasets. The threshold controls subcluster granularity and an optional final clustering stage combines subclusters.

Let `n` be observations and `p` input features. The algorithm learns a structure from `X` without a target label; each method optimizes a different objective, so scores from different methods are not interchangeable.


## 3. Working mechanism
1. Inspect observations, feature units, missingness, and geometry. 2. Fit transformations on training data only. 3. Fit the unsupervised estimator without reference labels. 4. Inspect method-appropriate diagnostics and review with domain experts. 5. Persist the full fitted preprocessing/model artifact and monitor it.


## 4. Algorithm from scratch (NumPy)
This compact reference shows K-means assignments and centroid updates; it is an educational contrast for the algorithm studied in this notebook.


In [ ]:
import numpy as np
scratch_X = np.array([[0., 0.], [.2, .1], [4., 4.], [4.2, 3.9]])
centers = scratch_X[[0, 2]].copy()
for _ in range(10):
    distances = np.linalg.norm(scratch_X[:, None, :] - centers[None, :, :], axis=2)
    groups = distances.argmin(axis=1)
    updated = np.vstack([scratch_X[groups == j].mean(axis=0) for j in range(len(centers))])
    if np.allclose(updated, centers):
        break
    centers = updated
print("reference assignments:", groups)
print("reference centers:\n", centers)


### Scratch dry run
Each observation is assigned to its nearest reference center, then centers are recomputed from assigned observations until they stop moving. This is the K-means update loop.


## 5. Dataset and business context
Clustering features are two-dimensional synthetic data; reference labels are for diagnostics only and are excluded from fitting. In a real project, document data provenance, units, privacy, and the business meaning of a useful structure.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
TASK = 'clustering'
from sklearn.datasets import make_blobs
X_raw, y_reference = make_blobs(n_samples=600, centers=4, n_features=2, cluster_std=0.75, random_state=42)
feature_names = ["feature_1", "feature_2"]
X_df = pd.DataFrame(X_raw, columns=feature_names)
X_train_raw, X_test_raw, y_train_reference, y_test_reference = train_test_split(X_df, y_reference, test_size=0.2, random_state=42)
feature_names = list(X_df.columns)
print("feature matrix:", X_df.shape)
print("reference labels are excluded from fitting")


## 6. Exploratory data analysis
Review dimensions, feature distributions, missingness, duplicates, and pairwise structure. A visualization can reveal scale issues or data errors, but it does not prove that clusters or latent factors are real.


In [ ]:
print("missing values:", int(X_df.isna().sum().sum()))
print("duplicate rows:", int(X_df.duplicated().sum()))
print(X_df.describe().T.head(10))
plt.scatter(X_train_raw.iloc[:, 0], X_train_raw.iloc[:, 1], s=12, alpha=.7)
plt.title("First two raw features (exploration only)")
plt.show()


## 7. Preprocessing without leakage
Split observations first. Fit the scaler using training rows and reuse it for test/new data. Scaling matters for distance and neighborhood methods. NMF requires non-negative input.


In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)
print("scaled train/test shapes:", X_train_scaled.shape, X_test_scaled.shape)


## 8. Baseline model
Use a deliberately simple reference method as a sanity check. The baseline does not use reference labels. Compare diagnostics and business usefulness rather than choosing an algorithm based on one score.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
baseline = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X_train_scaled)
print("simple K-means baseline silhouette:", round(silhouette_score(X_train_scaled, baseline.labels_), 3))


## 9. Detailed training
Fit the algorithm-specific estimator on transformed training features only. Reference labels are never passed to `fit` or `fit_predict`.


In [ ]:
from sklearn.cluster import Birch
model = Birch(n_clusters=4, threshold=0.5)
labels = model.fit_predict(X_train_scaled)
cluster_count = len(set(labels) - {-1})
noise_rate = float(np.mean(labels == -1)) if -1 in labels else 0.0
print("clusters (excluding noise):", cluster_count)
print("noise rate:", round(noise_rate, 3))


## 10. Evaluation
Silhouette is only defined when at least two non-noise clusters exist. ARI uses synthetic reference labels for diagnosis only. Neither metric alone establishes business usefulness.


In [ ]:
from sklearn.metrics import adjusted_rand_score, silhouette_score
valid = labels != -1
valid_labels = labels[valid]
if valid.sum() > 1 and 1 < np.unique(valid_labels).size < valid.sum():
    silhouette = float(silhouette_score(X_train_scaled[valid], valid_labels))
    print("training silhouette (non-noise rows):", round(silhouette, 3))
else:
    silhouette = None
    print("silhouette undefined: need at least two non-noise clusters")
print("synthetic-reference ARI (diagnostic only):", round(adjusted_rand_score(y_train_reference, labels), 3))
plt.scatter(X_train_raw.iloc[:, 0], X_train_raw.iloc[:, 1], c=labels, s=14, cmap="tab10")
plt.title("Training clusters; -1 denotes noise")
plt.show()


## 11. Stability and generalization
Unsupervised models have no universal train/test accuracy. Check whether structure is stable under random seeds, resampling, and reasonable perturbations. Use label-invariant scores for clustering, or neighborhood preservation for embeddings.


In [ ]:
from sklearn.base import clone
from sklearn.metrics import adjusted_rand_score
stability = []
for seed in range(5):
    candidate = clone(model)
    if "random_state" in candidate.get_params(deep=False):
        candidate.set_params(random_state=seed)
    stability.append(adjusted_rand_score(labels, candidate.fit_predict(X_train_scaled)))
print("ARI stability versus first fit:", np.round(stability, 3))
print("mean stability ARI:", round(float(np.mean(stability)), 3))


## 12. Hyperparameter optimization
Tune a small, meaningful parameter range using training-only diagnostics and domain context. Avoid repeatedly inspecting the final test partition. Clustering label identifiers can change between fits, so compare partitions with permutation-invariant measures.


In [ ]:
from sklearn.base import clone
from sklearn.metrics import silhouette_score
search_space = {
    "KMeans": ("n_clusters", [2, 3, 4, 5]),
    "AgglomerativeClustering": ("n_clusters", [2, 3, 4, 5]),
    "DBSCAN": ("eps", [0.15, 0.25, 0.4]),
    "OPTICS": ("min_samples", [5, 8, 12]),
    "GaussianMixture": ("n_components", [2, 3, 4, 5]),
    "MeanShift": ("bandwidth", [0.8, 1.1, 1.4]),
    "SpectralClustering": ("n_clusters", [2, 3, 4]),
    "Birch": ("threshold", [0.3, 0.5, 0.8]),
    "AffinityPropagation": ("preference", [-5.0, -2.0, 0.0]),
    "HDBSCAN": ("min_cluster_size", [10, 20, 30]),
}
parameter, candidates = search_space[model.__class__.__name__]
for value in candidates:
    candidate = clone(model).set_params(**{parameter: value})
    candidate_labels = candidate.fit_predict(X_train_scaled)
    keep = candidate_labels != -1
    unique = np.unique(candidate_labels[keep])
    score = silhouette_score(X_train_scaled[keep], candidate_labels[keep]) if keep.sum() > 1 and 1 < len(unique) < keep.sum() else None
    print(f"{parameter}={value}: silhouette={None if score is None else round(float(score), 3)}, clusters={len(unique)}")


## 13. Explainability and interpretation
Unsupervised structures are descriptive, not causal. Summaries can help characterize groups or dimensions aligned with an embedding. Validate interpretations with domain experts; do not overclaim from a visualization.


In [ ]:
import pandas as pd
profile = pd.DataFrame(X_train_raw, columns=feature_names).assign(cluster=labels)
print("feature means by cluster:")
print(profile.groupby("cluster").mean().round(2))


## 14. Advanced techniques
Assess cluster stability across initializations; stability supports reproducibility but does not prove business meaning.


In [ ]:
# Compare repeated fits using label-permutation-invariant cluster stability.
from sklearn.base import clone
from sklearn.metrics import adjusted_rand_score
stability = []
for seed in range(5):
    candidate = clone(model)
    if "random_state" in candidate.get_params(deep=False):
        candidate.set_params(random_state=seed)
    stability.append(adjusted_rand_score(labels, candidate.fit_predict(X_train_scaled)))
print("ARI stability versus first fit:", np.round(stability, 3))
print("mean stability ARI:", round(float(np.mean(stability)), 3))


## 15. Production implementation
Package the fitted scaler and estimator together with feature order and metadata so inference uses the same preprocessing. Pickle is executable serialization: load only trusted, version-controlled artifacts.


In [ ]:
import json
import pickle
from pathlib import Path
import sklearn
artifact_dir = Path("artifacts")
artifact_dir.mkdir(parents=True, exist_ok=True)
model_path = artifact_dir / "unsupervised_model.pkl"
bundle = {"scaler": scaler, "model": model, "feature_names": feature_names, "task": TASK}
with model_path.open("wb") as handle:
    pickle.dump(bundle, handle)
metadata = {"algorithm": model.__class__.__name__, "task": TASK, "sklearn_version": sklearn.__version__, "feature_count": len(feature_names), "artifact": str(model_path)}
(artifact_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
with model_path.open("rb") as handle:
    reloaded = pickle.load(handle)
print("saved:", model_path, "| reloaded features:", len(reloaded["feature_names"]))
print("smoke-test transformed shape:", reloaded["scaler"].transform(X_test_raw.iloc[:2]).shape)


## 16. Deployment
The FastAPI template loads the trusted bundle once at startup and validates feature names. Some clustering estimators only implement fit_predict and cannot label unseen production rows. This API returns HTTP 501 for those algorithms instead of silently refitting each request.


In [ ]:
import pickle
from pathlib import Path
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
with Path("artifacts/unsupervised_model.pkl").open("rb") as handle:
    bundle = pickle.load(handle)
app = FastAPI(title="Unsupervised ML API")
class AnalyzeRequest(BaseModel):
    features: dict[str, float]
@app.get("/health")
def health():
    return {"status": "ok"}
@app.post("/analyze")
def analyze(request: AnalyzeRequest):
    expected = bundle["feature_names"]
    if set(request.features) != set(expected):
        raise HTTPException(status_code=422, detail={"expected_features": expected})
    row = pd.DataFrame([[request.features[name] for name in expected]], columns=expected)
    transformed = bundle["scaler"].transform(row)
    if not hasattr(bundle["model"], "predict"):
        raise HTTPException(status_code=501, detail="This clustering algorithm has no native unseen-row predict method; deploy a separately validated assignment strategy.")
    labels = bundle["model"].predict(transformed)
    return {"labels": [int(value) for value in labels]}
    raise HTTPException(status_code=501, detail="No supported inference operation.")


## 17. MLOps
Record algorithm and parameter versions, training-data identity, preprocessing, diagnostics, artifact checksum, and runtime dependencies. Monitor feature drift, cluster-size/noise changes or embedding transform failures, latency, memory, and downstream usefulness. Retraining and rollout should be reviewed and reversible.


In [ ]:
import hashlib
import json
from datetime import datetime, timezone
training_fingerprint = hashlib.sha256(np.ascontiguousarray(X_train_scaled).tobytes()).hexdigest()
run_record = {
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "algorithm": model.__class__.__name__,
    "parameters": {key: str(value) for key, value in model.get_params(deep=False).items()},
    "training_rows": int(len(X_train_scaled)),
    "feature_count": int(X_train_scaled.shape[1]),
    "training_data_fingerprint": training_fingerprint,
    "artifact": str(model_path),
    "cluster_count": int(cluster_count), "noise_rate": noise_rate, "silhouette": silhouette,

}
(artifact_dir / "run_metadata.json").write_text(json.dumps(run_record, indent=2), encoding="utf-8")
print(run_record)


## 18. Performance optimization
Measure on representative data and hardware. Report fit time separately from online inference latency. Python memory tracing does not capture every native-library allocation. Reduce dimensions or sample data only when that preserves the intended analysis.


In [ ]:
import time
import tracemalloc
from sklearn.base import clone
sample_size = min(200, len(X_train_scaled))
benchmark = clone(model)
tracemalloc.start()
start = time.perf_counter()
benchmark.fit_predict(X_train_scaled[:sample_size])
fit_seconds = time.perf_counter() - start
_, peak_bytes = tracemalloc.get_traced_memory()
tracemalloc.stop()
print(f"fit time for {sample_size} rows: {fit_seconds:.4f} s")
print(f"fit throughput: {sample_size / fit_seconds:.1f} rows/s")
print(f"Python-traced peak: {peak_bytes / 1024:.1f} KB (native allocations may not be included)")


## 19. Production case study
**Workflow:** define the question and observation unit; verify point-in-time-correct features; establish preprocessing and validation; fit the method; review stability and diagnostics with domain experts; version model and data; deploy only a supported inference operation; monitor drift, output distribution, runtime, and downstream outcomes. Define rollback to a known-good artifact.


## 20. Interview preparation
1. Is this supervised? **No; target labels are not used in fitting.** 2. Why scale? **Distance and geometry can be scale-sensitive.** 3. What assumptions does it make? **See section 1.** 4. How do you select parameters? **Training-only diagnostics, stability, and domain context.** 5. How evaluate without labels? **Internal metrics and expert review; synthetic labels here are optional diagnostics.** 6. Can it process unseen rows? **Only with a supported predict/transform operation.** 7. How prevent leakage? **Fit preprocessing on training rows.** 8. What monitor? **Drift, output distribution, runtime, and usefulness.** 9. Can a 2D plot prove clusters? **No.** 10. How deploy safely? **Version trusted artifacts, validate schemas, and preserve rollback.**


## 21. Exercises
**Beginner:** change the seed and inspect results. **Intermediate:** compare two meaningful parameter settings and explain diagnostic changes. **Advanced:** test stability under resampling or compare another representation. **Production:** package preprocessing with the estimator and document whether unseen-row inference is supported. Never use reference labels to fit the unsupervised model.
